# 01 Cohort, target, leakage audit and EDA

LendingClub accepted loans, 36-month term, issued 2012-2015. This notebook builds the cohort and target, audits every column for leakage, and explores the training split.

## 1. Cohort and target

**Cohort.** LendingClub accepted loans issued 2012 to 2015 with a 36-month term. The data ends in 2018Q4, so 36-month loans from this window had time to reach a final outcome. 60-month loans or later vintages would have unresolved outcomes (censoring) and bias the default rate.

**Target.** default = 1 if loan_status is "Charged Off", default = 0 if "Fully Paid". Loans that are Current, Late or In Grace Period have an unknown outcome and are excluded.

**Result.** 589,635 loans are in the window; 589,488 are kept and 147 excluded (all issued in 2015: 72 Current, 61 Late 31-120 days, 9 In Grace Period, 5 Late 16-30 days). The overall default rate is 14.03% (82,728 defaults, 506,760 fully paid), which is mild imbalance.

**Time-based split by issue date.** Train 2012-2013 (143,892 loans, 12.70% default), validation 2014 (162,570 loans, 13.73%), test 2015 (283,026 loans, 14.89%). The test set is used once, at the very end.

**Limitation.** Only approved loans are observed (selection bias), and the 147 unresolved late-2015 loans are excluded, which has a negligible effect.

### Imports and settings

In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count())   # avoids a joblib core-count warning on Windows

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
for d in ["../data/processed", "../reports/figures", "../models"]:
    os.makedirs(d, exist_ok=True)

### Raw data path

In [ ]:
RAW_PATH = "../data/raw/accepted_2007_to_2018Q4.csv.gz"   # path to the raw LendingClub accepted file

### Read the column names and save the list

In [ ]:
all_cols = pd.read_csv(RAW_PATH, nrows=0).columns.tolist()
print("Number of columns:", len(all_cols))
pd.Series(all_cols, name="column").to_csv("../data/processed/all_columns.csv", index=False)
print(all_cols)

### Columns to load

In [ ]:
wanted_cols = [
    # identifiers and outcome
    "id", "loan_status", "issue_d", "term",
    # application-time information (candidate features)
    "loan_amnt", "installment", "emp_length", "home_ownership", "annual_inc",
    "verification_status", "purpose", "addr_state", "dti", "delinq_2yrs",
    "earliest_cr_line", "fico_range_low", "fico_range_high", "inq_last_6mths",
    "mths_since_last_delinq", "open_acc", "pub_rec", "revol_bal", "revol_util",
    "total_acc", "application_type", "mort_acc", "pub_rec_bankruptcies",
    # LendingClub's own risk output (used in feature set B and for comparison)
    "grade", "sub_grade", "int_rate",
    # post-issue columns: NEVER features; only for the business layer and the leakage demo
    "funded_amnt", "total_pymnt", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "last_pymnt_amnt", "last_fico_range_high", "last_fico_range_low",
]

missing = [c for c in wanted_cols if c not in all_cols]
print("Requested columns missing from the file:", missing)

### Load the data

In [ ]:
df = pd.read_csv(RAW_PATH, usecols=wanted_cols, low_memory=False)
print(df.shape)

### Remove non-loan rows at the bottom of the file

In [ ]:
print("Rows before:", len(df))
df = df[df["loan_status"].notna()].copy()
print("Rows after removing non-loan rows:", len(df))
print(df["loan_status"].value_counts())

### Parse dates and term

In [ ]:
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df["issue_year"] = df["issue_d"].dt.year
df["term"] = df["term"].str.strip()      # values look like " 36 months" with a space in front
print(df["term"].value_counts())
print(df["issue_d"].isna().sum(), "missing issue dates (should be 0)")

### Select the 36-month 2012-2015 cohort

In [ ]:
cohort_all = df[(df["term"] == "36 months") & (df["issue_year"].between(2012, 2015))].copy()
print("36-month loans issued 2012-2015:", len(cohort_all))
print(cohort_all["loan_status"].value_counts())
print()
print(pd.crosstab(cohort_all["issue_year"], cohort_all["loan_status"]))
del df   # frees memory

### Keep loans with a known outcome and create the target

In [ ]:
cohort = cohort_all[cohort_all["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
cohort["default"] = (cohort["loan_status"] == "Charged Off").astype(int)

n_total = len(cohort_all)
n_kept = len(cohort)
print(f"Loans in 36-month 2012-2015 window: {n_total}")
print(f"Kept (Fully Paid or Charged Off):  {n_kept}")
print(f"Excluded (other statuses):         {n_total - n_kept}")
print()
print(cohort["default"].value_counts())
print("Overall default rate:", round(cohort["default"].mean() * 100, 2), "%")

### Time-based split by issue year

In [ ]:
split_map = {2012: "train", 2013: "train", 2014: "valid", 2015: "test"}
cohort["split"] = cohort["issue_year"].map(split_map)

print(cohort["split"].value_counts())
print()
print(cohort.groupby("split")["issue_d"].agg(["min", "max"]))
print()
print(cohort.groupby("split")["default"].agg(["count", "sum", "mean"]))

### Default rate by vintage

In [ ]:
by_year = cohort.groupby("issue_year")["default"].agg(["count", "mean"])
by_year["default_rate_pct"] = (by_year["mean"] * 100).round(2)
print(by_year)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(by_year.index.astype(str), by_year["default_rate_pct"])
ax.set_xlabel("Issue year")
ax.set_ylabel("Default rate (%)")
ax.set_title("Default rate by issue year (36-month loans)")
for b, v in zip(bars, by_year["default_rate_pct"]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v}%", ha="center", va="bottom")
plt.tight_layout()
plt.savefig("../reports/figures/A1_default_rate_by_year.png", dpi=150)
plt.show()

**Takeaway:** The default rate dipped to 12.33% in 2013, then rose every year to 14.89% in 2015. Later vintages are riskier, so the population drifts over time, which is why we split by time.

### Default rate by issue quarter

In [ ]:
by_q = cohort.groupby(cohort["issue_d"].dt.to_period("Q"))["default"].mean() * 100
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(by_q.index.astype(str), by_q.values, marker="o")
ax.set_xticks(range(0, len(by_q), 2))
ax.set_xticklabels(by_q.index.astype(str)[::2], rotation=60)
ax.set_ylabel("Default rate (%)")
ax.set_title("Default rate by issue quarter")
plt.tight_layout()
plt.savefig("../reports/figures/A2_default_rate_by_quarter.png", dpi=150)
plt.show()

**Takeaway:** By quarter, the default rate fell from about 14% in mid-2012 to a low near 12% in 2013Q4, then climbed to a peak above 15% in 2015Q2 and stayed around 14.5 to 15%. The test period (2015) has a higher default rate than training, so test scores may differ from validation scores.

### Sanity checks

In [ ]:
print("Duplicate loan ids:", cohort["id"].duplicated().sum(), "(should be 0)")
print("Missing target:", cohort["default"].isna().sum(), "(should be 0)")
print("Missing split:", cohort["split"].isna().sum(), "(should be 0)")
print("Terms present:", cohort["term"].unique(), "(should be only '36 months')")
print("Statuses present:", cohort["loan_status"].unique())

### Save the cohort

In [ ]:
cohort = cohort.reset_index(drop=True)
cohort.to_parquet("../data/processed/cohort.parquet", index=False)
print("saved", cohort.shape)

## 2. Leakage audit
Rule: if a value would not exist on the day of approval, it cannot be a feature.

In [ ]:
try:
    cohort
    print("cohort already in memory:", cohort.shape)
except NameError:
    cohort = pd.read_parquet("../data/processed/cohort.parquet")
    print("cohort loaded from parquet:", cohort.shape)

### Decision for every column

In [ ]:
# Each row: (column, decision, category, reason)
audit_rows = [
    # ---- target, split helper, identifiers, free text ----
    ("loan_status", "Dropped", "Target", "This is the outcome we predict (source of the default label); only known after the loan ends."),
    ("issue_d", "Dropped", "Split helper", "Used only to split by time and to compute credit history at issue date; never a model feature."),
    ("id", "Dropped", "Identifier", "Unique loan ID with no predictive meaning; a model could memorize rows."),
    ("member_id", "Dropped", "Identifier", "Borrower ID with no predictive meaning."),
    ("url", "Dropped", "Identifier", "Link to the loan page; no predictive meaning."),
    ("desc", "Dropped", "Free text", "Free text written by the borrower; needs text mining and is outside project scope."),
    ("title", "Dropped", "Free text", "Free-text loan title; the clean categorical version is purpose."),
    ("emp_title", "Dropped", "Free text", "Free-text job title with thousands of different spellings (very high cardinality)."),
    ("zip_code", "Dropped", "Identifier", "Partial ZIP code with very high cardinality; overlaps addr_state and can act as a proxy for neighbourhood demographics."),

    # ---- application-time columns we keep ----
    ("loan_amnt", "Kept (A+B)", "Application", "Amount the borrower requested; known at application."),
    ("emp_length", "Kept (A+B)", "Application", "Self-reported years of employment; known at application."),
    ("home_ownership", "Kept (A+B)", "Application", "Rent, own or mortgage, as reported at application."),
    ("annual_inc", "Kept (A+B)", "Application", "Self-reported annual income at application."),
    ("verification_status", "Kept (A+B)", "Application", "Whether LendingClub verified the income; decided during underwriting, before funding."),
    ("purpose", "Kept (A+B)", "Application", "Reason for the loan, chosen by the borrower at application."),
    ("addr_state", "Kept (A+B)", "Application", "Borrower's state, given at application."),
    ("dti", "Kept (A+B)", "Application", "Debt-to-income ratio at application, excluding this loan."),
    ("delinq_2yrs", "Kept (A+B)", "Application", "Past-due incidents in the 2 years before the credit pull at application."),
    ("earliest_cr_line", "Kept (A+B)", "Application", "Month the borrower's first credit line opened; used to compute credit history length."),
    ("fico_range_low", "Kept (A+B)", "Application", "FICO score range at application (not the later last_fico_range columns)."),
    ("fico_range_high", "Kept (A+B)", "Application", "FICO score range at application (not the later last_fico_range columns)."),
    ("inq_last_6mths", "Kept (A+B)", "Application", "Credit inquiries in the 6 months before application."),
    ("mths_since_last_delinq", "Kept (A+B)", "Application", "Months since last delinquency at application; missing means none on record."),
    ("open_acc", "Kept (A+B)", "Application", "Open credit lines at application."),
    ("pub_rec", "Kept (A+B)", "Application", "Derogatory public records at application."),
    ("revol_bal", "Kept (A+B)", "Application", "Revolving credit balance at application."),
    ("revol_util", "Kept (A+B)", "Application", "Revolving credit utilization at application."),
    ("total_acc", "Kept (A+B)", "Application", "Total credit lines in the credit file at application."),
    ("application_type", "Kept (A+B)", "Application", "Individual or joint application; known at application."),
    ("mort_acc", "Kept (A+B)", "Application", "Number of mortgage accounts at application."),
    ("pub_rec_bankruptcies", "Kept (A+B)", "Application", "Public record bankruptcies at application."),

    # ---- LendingClub's own pricing / risk output ----
    ("grade", "Kept (B only)", "LendingClub risk output", "Assigned at approval, so not leakage, but it is LendingClub's own risk output; allowed only in set B to test whether our model adds anything beyond it."),
    ("int_rate", "Kept (B only)", "LendingClub risk output", "Set at approval from LendingClub's risk assessment; allowed only in set B."),
    ("installment", "Kept (B only)", "LendingClub risk output", "Monthly payment is computed from loan amount, interest rate and term, so it reveals the interest rate; allowed only in set B (see B.6 check)."),
    ("sub_grade", "Dropped", "LendingClub risk output", "Finer split of grade (A1 to G5); redundant with grade and int_rate, so not used in set A or B."),

    # ---- cohort filter ----
    ("term", "Dropped", "Cohort filter", "Known at application, but every loan in our cohort is 36 months, so it carries no information; used only to filter the cohort."),

    # ---- post-issue columns: LEAKAGE ----
    ("funded_amnt", "Dropped", "Post-issue (leakage)", "Amount investors actually funded, known only after funding; we use the requested loan_amnt. Used only in the profit calculation (notebook 04)."),
    ("funded_amnt_inv", "Dropped", "Post-issue (leakage)", "Portion of the funded amount from investors; known only after funding."),
    ("pymnt_plan", "Dropped", "Post-issue (leakage)", "Whether a payment plan was set up after the loan started; only arises after repayment trouble."),
    ("out_prncp", "Dropped", "Post-issue (leakage)", "Remaining principal owed; only exists once repayments have started."),
    ("out_prncp_inv", "Dropped", "Post-issue (leakage)", "Remaining principal owed to investors; only exists once repayments have started."),
    ("total_pymnt", "Dropped", "Post-issue (leakage)", "Total payments received so far; it is the loan's repayment outcome. Used only for profit in notebook 04."),
    ("total_pymnt_inv", "Dropped", "Post-issue (leakage)", "Total payments passed to investors; repayment outcome."),
    ("total_rec_prncp", "Dropped", "Post-issue (leakage)", "Principal repaid so far; defaulters repay less, so it reveals the outcome."),
    ("total_rec_int", "Dropped", "Post-issue (leakage)", "Interest paid so far; a record of repayment behaviour."),
    ("total_rec_late_fee", "Dropped", "Post-issue (leakage)", "Late fees paid; only exists if the borrower paid late."),
    ("recoveries", "Dropped", "Post-issue (leakage)", "Money recovered after a loan is charged off; can only be above zero for defaulted loans, so it gives away the label."),
    ("collection_recovery_fee", "Dropped", "Post-issue (leakage)", "Fee charged on recoveries; same problem as recoveries."),
    ("last_pymnt_d", "Dropped", "Post-issue (leakage)", "Date of the last payment received; a repayment record."),
    ("last_pymnt_amnt", "Dropped", "Post-issue (leakage)", "Amount of the last payment received; a repayment record."),
    ("next_pymnt_d", "Dropped", "Post-issue (leakage)", "Next scheduled payment date; exists only while the loan is active."),
    ("last_credit_pull_d", "Dropped", "Post-issue (leakage)", "Date LendingClub last pulled the borrower's credit, after issue."),
    ("last_fico_range_high", "Dropped", "Post-issue (leakage)", "Latest FICO score, pulled after issue; borrowers who defaulted show low scores."),
    ("last_fico_range_low", "Dropped", "Post-issue (leakage)", "Latest FICO score, pulled after issue; borrowers who defaulted show low scores."),

    # ---- other columns LendingClub sets, not borrower information ----
    ("initial_list_status", "Dropped", "Not borrower information", "Set by LendingClub when the loan is listed to investors; not a trait of the borrower."),
    ("policy_code", "Dropped", "Not borrower information", "Internal LendingClub product flag; not a trait of the borrower."),
    ("disbursement_method", "Dropped", "Not borrower information", "How the money is paid out, set at funding (after approval); not a trait of the borrower."),
]

# ---- groups of columns that share the same reason ----
bureau_unused = [
    "mths_since_last_record", "collections_12_mths_ex_med", "mths_since_last_major_derog", "acc_now_delinq",
    "tot_coll_amt", "tot_cur_bal", "open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m", "mths_since_rcnt_il",
    "total_bal_il", "il_util", "open_rv_12m", "open_rv_24m", "max_bal_bc", "all_util", "total_rev_hi_lim", "inq_fi",
    "total_cu_tl", "inq_last_12m", "acc_open_past_24mths", "avg_cur_bal", "bc_open_to_buy", "bc_util",
    "chargeoff_within_12_mths", "delinq_amnt", "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op",
    "mo_sin_rcnt_tl", "mths_since_recent_bc", "mths_since_recent_bc_dlq", "mths_since_recent_inq",
    "mths_since_recent_revol_delinq", "num_accts_ever_120_pd", "num_actv_bc_tl", "num_actv_rev_tl", "num_bc_sats",
    "num_bc_tl", "num_il_tl", "num_op_rev_tl", "num_rev_accts", "num_rev_tl_bal_gt_0", "num_sats", "num_tl_120dpd_2m",
    "num_tl_30dpd", "num_tl_90g_dpd_24m", "num_tl_op_past_12m", "pct_tl_nvr_dlq", "percent_bc_gt_75", "tax_liens",
    "tot_hi_cred_lim", "total_bal_ex_mort", "total_bc_limit", "total_il_high_credit_limit",
]
joint_cols = [
    "annual_inc_joint", "dti_joint", "verification_status_joint", "revol_bal_joint",
    "sec_app_fico_range_low", "sec_app_fico_range_high", "sec_app_earliest_cr_line", "sec_app_inq_last_6mths",
    "sec_app_mort_acc", "sec_app_open_acc", "sec_app_revol_util", "sec_app_open_act_il", "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths", "sec_app_collections_12_mths_ex_med", "sec_app_mths_since_last_major_derog",
]
hardship_cols = [
    "hardship_flag", "hardship_type", "hardship_reason", "hardship_status", "deferral_term", "hardship_amount",
    "hardship_start_date", "hardship_end_date", "payment_plan_start_date", "hardship_length", "hardship_dpd",
    "hardship_loan_status", "orig_projected_additional_accrued_interest", "hardship_payoff_balance_amount",
    "hardship_last_payment_amount",
]
settlement_cols = [
    "debt_settlement_flag", "debt_settlement_flag_date", "settlement_status", "settlement_date",
    "settlement_amount", "settlement_percentage", "settlement_term",
]

for c in bureau_unused:
    audit_rows.append((c, "Dropped", "Not used (application-time)",
        "Credit-bureau detail describing the borrower at application, but not in this project's chosen feature list (kept simple and explainable)."))
for c in joint_cols:
    audit_rows.append((c, "Dropped", "Not used (joint application)",
        "Filled only for joint applications (a second borrower); excluded for simplicity."))
for c in hardship_cols:
    audit_rows.append((c, "Dropped", "Post-issue (leakage)",
        "Hardship-program field, filled only after a borrower runs into trouble repaying."))
for c in settlement_cols:
    audit_rows.append((c, "Dropped", "Post-issue (leakage)",
        "Debt settlement happens after a borrower has stopped paying; strongly tied to default."))

print("rows written:", len(audit_rows))

### Build the audit table and check that all 151 columns are covered exactly once

In [ ]:
audit = pd.DataFrame(audit_rows, columns=["column", "decision", "category", "reason"])

all_cols = pd.read_csv("../data/processed/all_columns.csv")["column"].tolist()

not_covered = [c for c in all_cols if c not in set(audit["column"])]
not_in_file = sorted(set(audit["column"]) - set(all_cols))
duplicates = audit.loc[audit["column"].duplicated(), "column"].tolist()

print("Columns in the file:", len(all_cols))
print("Columns in the audit table:", len(audit))
print("In the file but NOT in the audit table:", not_covered)
print("In the audit table but NOT in the file:", not_in_file)
print("Duplicated in the audit table:", duplicates)

### Order like the original file, save, and summarize

In [ ]:
audit["order"] = audit["column"].map({c: i for i, c in enumerate(all_cols)})
audit = audit.sort_values("order").drop(columns="order").reset_index(drop=True)

audit.to_csv("../reports/leakage_audit.csv", index=False)

def to_md(df):
    lines = ["| Column | Decision | Category | Reason |", "|---|---|---|---|"]
    for _, r in df.iterrows():
        lines.append(f"| `{r['column']}` | {r['decision']} | {r['category']} | {r['reason']} |")
    return "\n".join(lines)

with open("../reports/leakage_audit.md", "w", encoding="utf-8") as f:
    f.write(to_md(audit))

print(audit["decision"].value_counts())
print()
print(audit.groupby(["decision", "category"]).size())

### Kept columns

In [ ]:
pd.set_option("display.max_colwidth", 200)
audit[audit["decision"].str.startswith("Kept")]

### Check kept columns against the LendingClub data dictionary

In [ ]:
dd = pd.read_excel("../data/raw/LCDataDictionary.xlsx")
print(dd.columns.tolist())
print(dd.shape)

In [ ]:
name_col, desc_col = dd.columns[0], dd.columns[1]
dd[name_col] = dd[name_col].astype(str).str.strip()

def show_desc(columns):
    for c in columns:
        row = dd[dd[name_col] == c]
        text = row[desc_col].values[0] if len(row) else "NOT FOUND in dictionary (search the file manually)"
        print(f"{c}\n   {text}\n")

kept_cols = audit.loc[audit["decision"].str.startswith("Kept"), "column"].tolist()
print("===== KEPT COLUMNS =====\n")
show_desc(kept_cols)

In [ ]:
print("===== KEY LEAKY COLUMNS =====\n")
show_desc(["total_pymnt", "total_rec_prncp", "recoveries", "last_pymnt_amnt",
           "last_fico_range_high", "last_fico_range_low", "out_prncp", "funded_amnt"])

### Why installment is only in feature set B

In [ ]:
ratio = cohort["installment"] / cohort["loan_amnt"]
print("Correlation between installment/loan_amnt and int_rate:", round(ratio.corr(cohort["int_rate"]), 4))

### Feature sets and clean modeling files

### Safety checks

In [ ]:
import sys
sys.path.append("../src")
from feature_sets import FEATURES_A_RAW, FEATURES_B_RAW

print("Set A columns:", len(FEATURES_A_RAW), "| Set B columns:", len(FEATURES_B_RAW))

dropped = set(audit.loc[audit["decision"] == "Dropped", "column"])
bad = [c for c in FEATURES_B_RAW if c in dropped]
assert len(bad) == 0, f"Dropped columns found in feature sets: {bad}"
assert not set(FEATURES_A_RAW) & {"grade", "sub_grade", "int_rate", "installment"}
print("OK: no dropped column is in the feature sets, and set A has no LendingClub risk output")

### Save modeling and business files

In [ ]:
id_cols = ["id", "issue_d", "issue_year", "split", "default"]
model_data = cohort[id_cols + FEATURES_B_RAW].copy()
model_data.to_parquet("../data/processed/model_data.parquet", index=False)

business_cols = ["id", "split", "default", "loan_amnt", "funded_amnt", "int_rate", "grade", "sub_grade",
                 "total_pymnt", "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
                 "recoveries", "collection_recovery_fee"]
business_data = cohort[business_cols].copy()
business_data.to_parquet("../data/processed/business_data.parquet", index=False)

post_issue = set(audit.loc[audit["category"] == "Post-issue (leakage)", "column"])
print("model_data shape:", model_data.shape)
print("Post-issue columns inside model_data (must be empty):", set(model_data.columns) & post_issue)
print("business_data shape:", business_data.shape)

### Leakage experiment

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

clean_cols = ["loan_amnt", "annual_inc", "dti", "delinq_2yrs", "fico_range_low", "fico_range_high",
              "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc"]
leaky_cols = ["total_pymnt", "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries",
              "collection_recovery_fee", "last_pymnt_amnt", "last_fico_range_high", "last_fico_range_low"]

train = cohort[cohort["split"] == "train"]
valid = cohort[cohort["split"] == "valid"]

def quick_auc(cols):
    model = HistGradientBoostingClassifier(max_iter=100, random_state=42)
    model.fit(train[cols], train["default"])
    p = model.predict_proba(valid[cols])[:, 1]
    return roc_auc_score(valid["default"], p)

auc_clean = quick_auc(clean_cols)
auc_leaky = quick_auc(clean_cols + leaky_cols)
print("Validation ROC-AUC, clean columns only :", round(auc_clean, 3))
print("Validation ROC-AUC, with leaky columns :", round(auc_leaky, 3))

rows = []
for c in leaky_cols:
    a = roc_auc_score(valid["default"], valid[c].fillna(0))
    rows.append((c, round(max(a, 1 - a), 3)))
single = pd.DataFrame(rows, columns=["leaky_column", "single_feature_AUC"]).sort_values("single_feature_AUC", ascending=False)
print()
print(single.to_string(index=False))

single.to_csv("../reports/leakage_demo_single_columns.csv", index=False)

print()
print(f"Validation ROC-AUC, clean columns only: {auc_clean:.3f} | with leaky columns: {auc_leaky:.3f}")

**Result (train 2012-2013, validated on 2014):** a gradient-boosting model reaches ROC-AUC 0.649 on application-time columns only, but 0.999 when post-issue columns (payments received, recoveries, last FICO) are added. The jump is not skill: those columns record what happened after approval, so all of them are excluded from the real models.

### Leakage audit summary
All 151 columns were reviewed one by one (table saved in `reports/leakage_audit.csv` and `reports/leakage_audit.md`) and checked against the LendingClub data dictionary. 25 columns are kept as features and 126 are dropped.

**Dropped:** post-issue columns (payments received, recoveries, last FICO, outstanding principal, hardship and debt-settlement fields), identifiers and free text, unused credit-bureau and joint-application fields, and fields set by LendingClub rather than the borrower.

**Feature set A (application-only, 22 columns)** excludes LendingClub's own risk output (grade, sub_grade, int_rate). **Feature set B (25 columns)** adds grade, int_rate and installment. Comparing A and B shows whether the model adds information beyond LendingClub's own assessment.

**Note on installment:** the monthly payment is calculated from loan amount, interest rate and term, so with a fixed 36-month term it exposes the interest rate (correlation of installment/loan_amnt with int_rate is ~1). It is therefore allowed only in set B.

**Files:** `data/processed/model_data.parquet` holds only issue date, split, target and features (no post-issue columns). `data/processed/business_data.parquet` holds the money columns for the profit analysis in notebook 04 only.

## 3. Exploratory analysis (training split only, 2012-2013)
Charts use only the training loans so that cleaning and modeling decisions are not influenced by validation or test data. The issue-year view was done in section 1.

### Load the modeling file and set up plotting

In [ ]:
import sys
sys.path.append("../src")
import seaborn as sns
from IPython.display import Markdown, display
from feature_sets import FEATURES_A_RAW, FEATURES_B_RAW

sns.set_theme(style="whitegrid")

model_data = pd.read_parquet("../data/processed/model_data.parquet")
train = model_data[model_data["split"] == "train"].copy()

BASE_RATE = train["default"].mean() * 100
print("model_data:", model_data.shape, "| train:", train.shape)
print("Training default rate:", round(BASE_RATE, 2), "%")

### Plotting helper

In [ ]:
def default_rate_plot(df, col, title, fname, sort_by_rate=False, min_count=1, figsize=(8, 4), rotate=0):
    g = df.groupby(col, observed=True)["default"].agg(["count", "mean"])
    g = g[g["count"] >= min_count].copy()
    g["rate"] = g["mean"] * 100
    if sort_by_rate:
        g = g.sort_values("rate")
    print(g[["count", "rate"]].round(2).to_string())

    fig, ax = plt.subplots(figsize=figsize)
    labels = [str(i) for i in g.index]
    bars = ax.bar(labels, g["rate"])
    ax.axhline(BASE_RATE, color="red", linestyle="--", linewidth=1,
               label=f"Overall train rate {BASE_RATE:.1f}%")
    ax.set_ylabel("Default rate (%)")
    ax.set_xlabel(col)
    ax.set_title(title)
    ax.legend()
    plt.xticks(rotation=rotate)
    if len(g) <= 15:
        for b, v in zip(bars, g["rate"]):
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.savefig(f"../reports/figures/{fname}", dpi=150)
    plt.show()

    lo, hi = g["rate"].idxmin(), g["rate"].idxmax()
    text = (f"**Takeaway:** default rate ranges from {g['rate'].min():.1f}% ({lo}) "
            f"to {g['rate'].max():.1f}% ({hi}) across {col} groups")
    if not sort_by_rate:
        if g["rate"].is_monotonic_increasing:
            text += ", rising steadily from the first group to the last."
        elif g["rate"].is_monotonic_decreasing:
            text += ", falling steadily from the first group to the last."
        else:
            text += ", without a perfectly steady trend."
    else:
        text += "."
    display(Markdown(text))

### Create DTI, FICO and income bands

In [ ]:
train["dti_band"] = pd.cut(train["dti"], bins=[-np.inf, 10, 15, 20, 25, 30, 35, 40, np.inf],
                           labels=["<10", "10-15", "15-20", "20-25", "25-30", "30-35", "35-40", "40+"], right=False)

train["fico_mid"] = (train["fico_range_low"] + train["fico_range_high"]) / 2
train["fico_band"] = pd.cut(train["fico_mid"], bins=[0, 680, 700, 720, 740, 760, 780, np.inf],
                            labels=["<680", "680-699", "700-719", "720-739", "740-759", "760-779", "780+"], right=False)

train["income_band"] = pd.cut(train["annual_inc"], bins=[0, 30000, 45000, 60000, 80000, 100000, 150000, np.inf],
                              labels=["<30k", "30-45k", "45-60k", "60-80k", "80-100k", "100-150k", "150k+"], right=False)

print(train[["dti_band", "fico_band", "income_band"]].isna().sum())

### Default rate by LendingClub grade

In [ ]:
default_rate_plot(train, "grade", "Default rate by LendingClub grade", "C1_default_by_grade.png")

**Note:** grades A to F rise steadily from 5.6% to 25.6%. Grade G dips to 20.5% only because it has just 39 loans in the training split, so its rate is not reliable.

### Default rate by DTI band

In [ ]:
default_rate_plot(train, "dti_band", "Default rate by debt-to-income band", "C2_default_by_dti.png")

### Default rate by FICO band

In [ ]:
default_rate_plot(train, "fico_band", "Default rate by FICO band", "C3_default_by_fico.png")

### Default rate by income band

In [ ]:
default_rate_plot(train, "income_band", "Default rate by annual income band", "C4_default_by_income.png")

### Default rate by purpose (groups with fewer than 300 loans are hidden because their rates are too noisy to trust)

In [ ]:
default_rate_plot(train, "purpose", "Default rate by loan purpose", "C5_default_by_purpose.png",
                  sort_by_rate=True, min_count=300, figsize=(10, 4), rotate=60)

### Default rate by home ownership

In [ ]:
default_rate_plot(train, "home_ownership", "Default rate by home ownership", "C6_default_by_home.png",
                  sort_by_rate=True, min_count=300)

### Default rate by state (states with fewer than 500 training loans hidden)

In [ ]:
default_rate_plot(train, "addr_state", "Default rate by state", "C7_default_by_state.png",
                  sort_by_rate=True, min_count=500, figsize=(14, 4.5), rotate=90)

### Missingness

In [ ]:
miss = train[FEATURES_B_RAW].isna().mean().mul(100).sort_values(ascending=False)
print(miss.round(2).to_string())

miss_nonzero = miss[miss > 0]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(miss_nonzero.index[::-1], miss_nonzero.values[::-1])
ax.set_xlabel("Missing values (%)")
ax.set_title("Missing values per feature (training split)")
plt.tight_layout()
plt.savefig("../reports/figures/C8_missingness.png", dpi=150)
plt.show()

top = miss_nonzero.head(3)
parts = ", ".join(f"{name} ({val:.1f}%)" for name, val in top.items())
display(Markdown(f"**Takeaway:** the most incomplete features are {parts}. "
                 "For mths_since_last_delinq a blank means the borrower has no delinquency on record, so the blank itself is informative."))

### Missingness by year (data-collection check, so all years are used)

In [ ]:
check_cols = list(miss_nonzero.index[:5])
by_year_miss = model_data.groupby("issue_year")[check_cols].apply(lambda d: d.isna().mean() * 100).round(2)
print(by_year_miss)

### Distributions of skewed variables

In [ ]:
skew_cols = ["annual_inc", "revol_bal", "dti", "revol_util", "loan_amnt", "total_acc", "open_acc"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), skew_cols):
    s = train[c].dropna()
    upper = s.quantile(0.995)
    ax.hist(s[s <= upper], bins=50)
    ax.set_title(f"{c} (skew {s.skew():.1f})")
for ax in axes.ravel()[len(skew_cols):]:
    ax.set_visible(False)
plt.suptitle("Distributions of numeric features (top 0.5% of values hidden for readability)")
plt.tight_layout()
plt.savefig("../reports/figures/C9_distributions.png", dpi=150)
plt.show()

skews = train[skew_cols].skew().sort_values(ascending=False)
heavy = ", ".join(f"{k} ({v:.1f})" for k, v in skews.items() if v > 1)
display(Markdown(f"**Takeaway:** strongly right-skewed features (skew above 1): {heavy}. "
                 "These need a log transform and/or capping of extreme values in the feature pipeline."))

#### Size of the extremes (motivates capping in the feature pipeline)

In [ ]:
print(train[skew_cols].describe(percentiles=[0.01, 0.5, 0.99, 0.999]).T.round(2).to_string())
print()
print("revol_util above 100:", (train["revol_util"] > 100).sum())
print("dti below 0:", (train["dti"] < 0).sum(), "| dti above 100:", (train["dti"] > 100).sum())
print("annual_inc equal to 0:", (train["annual_inc"] == 0).sum())

#### Income before and after a log transform

In [ ]:
s = train["annual_inc"].dropna()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(s[s <= s.quantile(0.995)], bins=50)
axes[0].set_title(f"annual_inc raw (skew {s.skew():.1f})")
axes[1].hist(np.log1p(s), bins=50)
axes[1].set_title(f"log(1 + annual_inc) (skew {np.log1p(s).skew():.1f})")
plt.tight_layout()
plt.savefig("../reports/figures/C10_income_log.png", dpi=150)
plt.show()
display(Markdown(f"**Takeaway:** a log transform reduces the skew of annual income from {s.skew():.1f} to {np.log1p(s).skew():.1f}, giving a much more balanced shape for modeling."))

#### Correlation heatmap

In [ ]:
num_cols = ["loan_amnt", "annual_inc", "dti", "delinq_2yrs", "fico_range_low", "fico_range_high",
            "inq_last_6mths", "mths_since_last_delinq", "open_acc", "pub_rec", "revol_bal", "revol_util",
            "total_acc", "mort_acc", "pub_rec_bankruptcies", "int_rate", "installment"]

corr = train[num_cols + ["default"]].corr(method="spearman")

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7}, ax=ax)
ax.set_title("Spearman correlation heatmap (training split)")
plt.tight_layout()
plt.savefig("../reports/figures/C11_correlation_heatmap.png", dpi=150)
plt.show()

target_corr = corr["default"].drop("default")
target_corr = target_corr.reindex(target_corr.abs().sort_values(ascending=False).index)

feat_corr = corr.drop(index="default", columns="default")
pairs = feat_corr.where(np.triu(np.ones(feat_corr.shape), k=1).astype(bool)).stack()
pairs = pairs.reindex(pairs.abs().sort_values(ascending=False).index)

print("Correlation with default (strongest first):")
print(target_corr.round(3).head(8).to_string())
print()
print("Most correlated feature pairs:")
print(pairs.round(3).head(6).to_string())

top3 = ", ".join(f"{k} ({v:+.2f})" for k, v in target_corr.head(3).items())
p1, p2 = pairs.index[0], pairs.index[1]
display(Markdown(f"**Takeaway:** the features most correlated with default are {top3}; all correlations with default are modest, which fits a realistic AUC around 0.65-0.70. "
                 f"The most redundant feature pairs are {p1[0]} / {p1[1]} ({pairs.iloc[0]:.2f}) and {p2[0]} / {p2[1]} ({pairs.iloc[1]:.2f}), so some features carry overlapping information."))

### EDA summary
Default rates were examined by grade, DTI, FICO, income, purpose, home ownership and state using the training split only (issue year was covered in section 1). Missing values, skewed distributions and correlations were reviewed. Each chart has its takeaway directly below it.

**Decisions carried into the feature pipeline (cleaning and features):**
1. Convert `emp_length` text (such as "10+ years") into a number.
2. Convert `earliest_cr_line` into `credit_history_years` measured at the issue date.
3. Replace the two FICO columns with their midpoint `fico_mid` (they are almost perfectly correlated).
4. Use `log(annual_inc)` and ratios such as loan-to-income to handle skew.
5. Cap extreme values at percentiles computed on the training set only.
6. Add missing-value indicators (for example for `mths_since_last_delinq`, where blank means no delinquency on record).
7. Group rare categories and one-hot encode categorical features.